# RAPID hydrograph review

Review observed gauge events, refreshed model baselines and individual collapse responses. This notebook diagnoses timing/shape and domain changes; predictor benchmarking remains separate. Updated 5 October 2026.

**Updated source:** active published Cunge registries under `network_variants/output`, including gauge overlays, parent/child plots and timing tables. No historical-refresh fallback. Peak times are native UTC. Earlier resistance values are not transferred across releases. Maps highlight saved domain membership only after validating its topology against current parent/child graphs; ZOOM_TO_DOMAIN=True zooms to those links. The commented legacy investigation cells below require a current-domain adapter before reactivation.

## Purpose and reading order

The original notebook only inspected zero/positive/negative collapse timing groups and their selected resistance domains. It did not validate against gauges and used historical runs. This version reads the October hydraulic refresh, adds observed/model overlays on verified actual UTC, and preserves those three inspection groups. A sampled zero means a timestamp tie, not an identical hydrograph. Interpolated timing is shown as a separate diagnostic; no per-river calibration or peak alignment is applied.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "RAPID" / "src" / "rapid_tools").exists():
            return candidate
    raise FileNotFoundError("Could not locate the river-hierarchy repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RAPID_SRC = REPO_ROOT / "RAPID" / "src"
OUTPUT_ROOT = REPO_ROOT / "network_variants" / "output"
if str(RAPID_SRC) not in sys.path:
    sys.path.insert(0, str(RAPID_SRC))

import importlib
import rapid_tools.current_review as current_review
importlib.reload(current_review)
from rapid_tools.current_review import (
    build_current_analysis_frame, gauge_overlay, select_network_pairs,
    parent_child_hydrograph_figure, parent_child_network_figure,
    collapse_hydrograph_overlay,
)
import plotly.graph_objects as go
import json


## Configuration and input assembly

The cells below load the completed experiments and form the single analysis population used by every investigation section. Run them before using the review cells.


In [3]:
EXAMPLE_IDS = ["02","03","05","11","13","15","21","22","37","41","45","46","48","54","55","56"]
# Read current published registries; unavailable examples are reported, never loaded from an older release.
PRIMARY_Y_METRIC = "lag_to_inflow_peak_pct_shift_vs_base"
# This percentage uses sampled peak lag; reviewed_feature_shift_minutes separately retains 54's centre.
# Historical resistance/domain metrics are not mixed into current timing results.


## Build the validated analysis population

Only states with the required RAPID and selected-domain equivalent-resistance records enter `plot_df`. The following cell is the shared input contract for all three response classes.


In [4]:
analysis_df = build_current_analysis_frame(EXAMPLE_IDS, output_root=OUTPUT_ROOT)
analysis_df["primary_pair_complete"] = np.isfinite(analysis_df[PRIMARY_Y_METRIC])
coverage = analysis_df.groupby(["example_id", "mode"]).agg(
    states=("state_id", "nunique"), unique_hydrographs=("fingerprint", "nunique"),
    timing_available=(PRIMARY_Y_METRIC, lambda s: np.isfinite(s).sum()),
).reset_index()
display(coverage)
print("Source: active published Cunge rapid_run_registry.csv; historical refresh is not loaded.")
print("Unique child comparisons:", len(analysis_df.loc[~analysis_df.technical_alias]))
display(analysis_df[["example_id","mode","state_id","sampled_peak_shift_minutes",
    "reviewed_feature_shift_minutes","peak_total_change_pct","parent_peak_utc","child_peak_utc",
    "registry_path"]].head())


/Users/6256481/Code/river-hierarchy/RAPID/src/rapid_tools/current_review.py:24: UserWarning: 11/indep: registry has historical schema; excluded from current review.
  warnings.warn(f'{ex}/{family}: registry has historical schema; excluded from current review.')
/Users/6256481/Code/river-hierarchy/RAPID/src/rapid_tools/current_review.py:24: UserWarning: 11/compound_bubbles: registry has historical schema; excluded from current review.
  warnings.warn(f'{ex}/{family}: registry has historical schema; excluded from current review.')
/Users/6256481/Code/river-hierarchy/RAPID/src/rapid_tools/current_review.py:30: UserWarning: 11: no current published registry; excluded (no historical fallback).
  warnings.warn(f'{ex}: no current published registry; excluded (no historical fallback).')


,example_id,mode,states,unique_hydrographs,timing_available
0,02,compound,5,5,5
1,02,indep,26,26,26
2,02,island,26,26,26
3,03,compound,7,7,7
4,03,indep,31,31,31
5,03,island,31,31,31
6,05,compound,8,8,8
7,05,indep,43,43,43
8,05,island,43,43,43
9,13,compound,10,10,10


Source: active published Cunge rapid_run_registry.csv; historical refresh is not loaded.
Unique child comparisons: 2172


,example_id,mode,state_id,sampled_peak_shift_minutes,reviewed_feature_shift_minutes,peak_total_change_pct,parent_peak_utc,child_peak_utc,registry_path
0,02,indep,S001_unit_2,-7.0,-7.0,-0.008356,2025-08-01T22:59:30+00:00,2025-08-01T22:52:30+00:00,/Users/6256481/Code/river-hierarchy/network_va...
1,02,indep,S002_unit_18,1.0,1.0,0.029712,2025-08-01T22:59:30+00:00,2025-08-01T23:00:30+00:00,/Users/6256481/Code/river-hierarchy/network_va...
2,02,indep,S003_unit_17,3.0,3.0,0.373322,2025-08-01T22:59:30+00:00,2025-08-01T23:02:30+00:00,/Users/6256481/Code/river-hierarchy/network_va...
3,02,indep,S004_unit_5,0.0,0.0,-0.048360,2025-08-01T22:59:30+00:00,2025-08-01T22:59:30+00:00,/Users/6256481/Code/river-hierarchy/network_va...
4,02,indep,S005_unit_25,-1.0,-1.0,-0.002408,2025-08-01T22:59:30+00:00,2025-08-01T22:58:30+00:00,/Users/6256481/Code/river-hierarchy/network_va...


In [4]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

# ── Settings ───────────────────────────────────────────────────────────
TIMING_COLUMN = "sampled_peak_shift_minutes"
# Use "reviewed_feature_shift_minutes" to use the centre timing for 54.

N_BINS = 80
X_RANGE = None           # Example: [-30, 30] to zoom; calculations use all data.
KDE_BANDWIDTH = "scott"

GROUPS = [
    "Simple",
    "Nested",
    "Compound member",
    "Island",
    "Full compound",
]

COLORS = {
    "Simple": "#2878B5",
    "Nested": "#9467BD",
    "Compound member": "#D95F02",
    "Island": "#2CA02C",
    "Full compound": "#D62728",
}

# ── Current responses: remove technical aliases ────────────────────────
df = analysis_df.loc[~analysis_df["technical_alias"]].copy()
df["example_id"] = df["example_id"].astype(str).str.zfill(2)

# Audited hierarchy membership for this release.
membership = pd.read_parquet(
    REPO_ROOT
    / "RAPID/outputs/geometry_response_foundation_20261012T120140"
    / "canonical_pairs.parquet"
).rename(columns={"example": "example_id"})

membership["example_id"] = (
    membership["example_id"].astype(str).str.zfill(2)
)

membership_columns = ["any_nested_unit", "any_compound_member"]

# Allow this cell to run even if membership columns were added previously.
df = df.drop(columns=membership_columns, errors="ignore").merge(
    membership[["example_id", "fingerprint", *membership_columns]],
    on=["example_id", "fingerprint"],
    how="left",
    validate="one_to_one",
)

if df[membership_columns].isna().any().any():
    raise ValueError(
        "Some current states lack audited hierarchy membership. "
        "Check that the membership snapshot matches the current release."
    )

# Mutually exclusive classification:
# independent island cases → nested → compound member → simple.
# Island aliases of independent units have already been removed.
# Mutually exclusive groups. Full-compound cases take priority.
df["group"] = np.select(
    [
        df["mode"].eq("compound"),
        df["mode"].eq("island"),
        df["any_nested_unit"].eq(True),
        df["any_compound_member"].eq(True),
    ],
    ["Full compound", "Island", "Nested", "Compound member"],
    default="Simple",
)

df["shift_min"] = pd.to_numeric(df[TIMING_COLUMN], errors="coerce")
valid = np.isfinite(df["shift_min"])
print(f"Missing/nonfinite timings excluded: {(~valid).sum()}")
df = df.loc[valid].copy()

if df.empty:
    raise ValueError("No valid peak shifts remain.")

# ── Shared bins and KDE evaluation grid ────────────────────────────────
lo = float(df["shift_min"].min())
hi = float(df["shift_min"].max())

if lo == hi:
    lo, hi = lo - 1, hi + 1

bin_width = (hi - lo) / N_BINS
grid = np.linspace(lo, hi, 3000)

fig = go.Figure()

for group in GROUPS:
    values = df.loc[df["group"].eq(group), "shift_min"].to_numpy()
    if not len(values):
        continue

    label = f"{group} (n={len(values)})"
    has_kde = len(values) > 1 and np.std(values) > 0

    fig.add_trace(go.Histogram(
        x=values,
        xbins=dict(start=lo, end=hi, size=bin_width),
        histnorm="probability density",
        marker_color=COLORS[group],
        opacity=0.18,
        name=label,
        legendgroup=group,
        showlegend=not has_kde,
        hovertemplate=(
            f"{group}<br>"
            "Shift bin: %{x} min<br>"
            "Density: %{y:.4g}<extra></extra>"
        ),
    ))

    if has_kde:
        kde = gaussian_kde(values, bw_method=KDE_BANDWIDTH)

        fig.add_trace(go.Scatter(
            x=grid,
            y=kde(grid),
            mode="lines",
            line=dict(color=COLORS[group], width=2.5),
            name=label,
            legendgroup=group,
            hovertemplate=(
                f"{group}<br>"
                "Shift: %{x:.2f} min<br>"
                "KDE density: %{y:.4g}<extra></extra>"
            ),
        ))

fig.add_vline(
    x=0,
    line_color="black",
    line_dash="dash",
    line_width=1,
)

timing_label = (
    "Sampled peak"
    if TIMING_COLUMN == "sampled_peak_shift_minutes"
    else "Reviewed crest feature"
)

fig.update_layout(
    title=f"{timing_label} timing changes by collapse group",
    template="plotly_white",
    barmode="overlay",
    width=1050,
    height=550,
    xaxis_title="Child − parent timing (minutes): earlier ← 0 → later",
    yaxis_title="Probability density (per minute)",
    legend=dict(
        title="Collapse group",
        groupclick="togglegroup",
    ),
    hovermode="closest",
)

if X_RANGE is not None:
    fig.update_xaxes(range=X_RANGE)

fig.show()

# ── Counts and timing summary ──────────────────────────────────────────
summary = (
    df.groupby("group")["shift_min"]
      .agg(
          n="size",
          earlier=lambda x: (x < 0).sum(),
          unchanged=lambda x: (x == 0).sum(),
          later=lambda x: (x > 0).sum(),
          median_minutes="median",
          minimum_minutes="min",
          maximum_minutes="max",
      )
      .reindex(GROUPS)
      .dropna(subset=["n"])
)

display(summary)

Missing/nonfinite timings excluded: 0


,n,earlier,unchanged,later,median_minutes,minimum_minutes,maximum_minutes
group,,,,,,,
Simple,258,241,8,9,-6.0,-47.0,6.0
Nested,738,537,148,53,-1.0,-137.0,77.0
Compound member,750,564,120,66,-2.0,-84.0,17.0
Island,269,145,45,79,-1.0,-28.0,127.0
Full compound,157,154,0,3,-21.0,-525.0,13.0


## Apply the population filter

This applies the configured edge-compound filter and produces `plot_df`, the common state-level table used below. No association screening or model-selection figures are retained here.


In [5]:
plot_df = analysis_df.loc[~analysis_df.technical_alias & analysis_df.primary_pair_complete].copy()
print(f"Plotting {len(plot_df)} states from {plot_df['example_id'].nunique()} examples.")
# No historical domain/edge-compound filter: current matched-domain metadata must first be validated.


Plotting 2172 states from 15 examples.


## Gauge hydrographs and optional model overlay

Select any or all cases. `INCLUDE_MODEL=False` gives gauges only; plot buttons also toggle model visibility. UTC is recovered from the original forcing after verifying identical time/flow arrays. No alignment to the observed downstream peak is performed. Set `NORMALIZE=False` to compare actual discharge; normalization is explicitly shape-only. Peak tables refer to the quality-sheet observation window; a model peak outside it requires inspecting the full model event.

If the model peak lies beyond the gauge review window, the display extends to include it; a dotted line marks the end of that window. Observations and peak tables retain the selected gauge window.

In [6]:
GAUGE_EXAMPLES = ["37"]  # Current case; use sorted(analysis_df.example_id.unique()) for all networks.
INCLUDE_MODEL = True
NORMALIZE = False
gauge_peak_tables = []
for example in GAUGE_EXAMPLES:
    fig, table = gauge_overlay(example, analysis=analysis_df, include_model=INCLUDE_MODEL, normalize=NORMALIZE)
    fig.show()
    gauge_peak_tables.append(table)
display(pd.concat(gauge_peak_tables, ignore_index=True))

,example_id,series,peak_utc,peak_cms
0,37,Prepared upstream gauge,2014-06-27 18:29:30+00:00,2675.658834
1,37,Prepared downstream gauge,2014-06-28 11:14:30+00:00,4699.652639
2,37,Applied forcing,2014-06-27 18:29:30+00:00,2675.658834
3,37,Current parent outlet,2014-06-28 09:15:30+00:00,2628.601639


## Parent and child network states

Case-by-case hydrograph interpretation is recorded in **`RAPID/reviews/hydrograph_match_review.xlsx`** (review dated 5 October 2026). Use this section to inspect the actual parent and child geometry.

Choose one example and optionally a subset of `(mode, state_id)` keys from the table. `UNIT_SELECTION = None` selects all eligible states; **at most `MAX_NETWORK_PAIRS` (default 6) are drawn**. Any omitted states are counted explicitly. An empty list selects none. Default browsing excludes technical aliases. Explicit `(mode, state_id)` selections may include aliases; the available-unit table labels them with `technical_alias`. The modes are `indep`, `compound`, and `island`.

Each pair highlights the selected matched domain on the current parent and child networks. Saved domain membership is checked against current topology. ZOOM_TO_DOMAIN=True zooms both maps to the shared domain extent; False shows network context. A containing/enclosing domain is labelled explicitly when applicable.


Each selected pair also has a **parent–child hydrograph** using the same example, unit selection and upper limit. These are whole-network outlet responses to the selected collapse, not local unit-outlet discharges. Blue is the parent; dashed red is the child. The shared input forcing is optional (`SHOW_INPUT_FORCING`). Set `SHOW_NETWORK_MAPS=False` to show only hydrographs, or `SHOW_HYDROGRAPHS=False` for only maps. Saved UTC timestamps, discharge magnitudes and the full routed event are preserved; use Plotly zoom to inspect overlapping peaks.


In [50]:
# Reload the shared selector as well as the current-output plotting helpers.
import importlib
import rapid_tools.hydrograph_review as _selection_helpers
import rapid_tools.current_review as _current_review
importlib.reload(_selection_helpers)
import rapid_tools.current_domains as _current_domains
importlib.reload(_current_domains)
importlib.reload(_current_review)
from rapid_tools.current_review import select_network_pairs, parent_child_network_figure, parent_child_hydrograph_figure

NETWORK_EXAMPLE = "37"
NETWORK_MODES = None  # All modes; or ["island"] to filter the table before choosing a row.
MAX_NETWORK_PAIRS = 1
ZOOM_TO_DOMAIN = True
SHOW_NETWORK_MAPS = True
SHOW_HYDROGRAPHS = True
SHOW_INPUT_FORCING = True

available_units = analysis_df.loc[
    analysis_df.example_id.eq(NETWORK_EXAMPLE),
    ["mode", "state_id", "technical_alias", "target_id", "sampled_peak_shift_minutes", "reviewed_feature_shift_minutes", "peak_total_change_pct"],
].drop_duplicates(["mode", "state_id"]).sort_values(["mode", "state_id"])
if NETWORK_MODES is not None:
    available_units = available_units[available_units["mode"].isin(NETWORK_MODES)]
print(f"{len(available_units)} available units/states; first 30 keys shown below. Full table: available_units.")
# display(available_units.head(30))

# Select the third row of the freshly built table (zero-based position 2).
# Each key needs BOTH mode and state_id, even when NETWORK_MODES is set.
selected_unit = available_units.iloc[2]
# UNIT_SELECTION = [(selected_unit["mode"], selected_unit["state_id"])]
UNIT_SELECTION = [
    # ("island", "island_15_047"),
    # ("indep", "S038_unit_4"),
    # ("indep", "S130_unit_126"),
    ("compound", "CB002_bubble_2")

]
# UNIT_SELECTION =None

# Alternatively: UNIT_SELECTION = None for all eligible states, or [] for none.
selected_networks, omitted_networks = select_network_pairs(
    analysis_df, NETWORK_EXAMPLE, units=UNIT_SELECTION,
    modes=NETWORK_MODES, max_pairs=MAX_NETWORK_PAIRS,
)
print(f"Plotting {len(selected_networks)} pairs. Upper limit: {MAX_NETWORK_PAIRS}. Omitted by limit: {len(omitted_networks)}.")
for _, selected_network in selected_networks.iterrows():
    display(selected_network[["parent_peak_utc","child_peak_utc","sampled_peak_shift_minutes",
        "reviewed_feature_shift_minutes","peak_total_change_pct","parent_outlet_parquet","outlet_parquet"]].to_frame("current output"))
    if SHOW_NETWORK_MAPS:
        try:
            # parent_child_network_figure(selected_network, zoom_to_domain=ZOOM_TO_DOMAIN).show()
            parent_child_network_figure(selected_network, zoom_to_domain=ZOOM_TO_DOMAIN,
                                        layout='overlay', show_children=False).show()
        except (ValueError, FileNotFoundError) as exc:
            print(f"Map unavailable (hydrograph review continues): {exc}")
    if SHOW_HYDROGRAPHS:
        parent_child_hydrograph_figure(selected_network,show_title=False,
        show_forcing=False,show_downstream_gauge=False,
        show_parent=True,show_children=False,
        x_range=(pd.Timestamp("2014-06-28 03:00"), "2014-06-28 15:00"),
        y_range=(2610,2630)).show()


652 available units/states; first 30 keys shown below. Full table: available_units.
Plotting 1 pairs. Upper limit: 1. Omitted by limit: 0.


,current output
parent_peak_utc,2014-06-28T09:15:30+00:00
child_peak_utc,2014-06-28T08:53:30+00:00
sampled_peak_shift_minutes,-22.0
reviewed_feature_shift_minutes,-22.0
peak_total_change_pct,0.00481
parent_outlet_parquet,/Volumes/PhD/river_hierarchy/network_variants/...
outlet_parquet,/Volumes/PhD/river_hierarchy/network_variants/...


## Multiple collapse hydrographs in one graph

Uses the same `NETWORK_EXAMPLE`, `NETWORK_MODES`, `UNIT_SELECTION`, `MAX_NETWORK_PAIRS` and `SHOW_INPUT_FORCING` controls above. The parent is drawn once, with one line per selected collapse. `None` selects all eligible states up to the stated limit; click legend entries to isolate curves. These are whole-network outlet hydrographs on verified shared clocks, without peak alignment. The selection table preserves mode/state keys, including deliberately selected aliases; coincident lines are possible.


In [8]:
# Reload helpers so this cell also works in an already-running kernel.
import importlib
import rapid_tools.current_review as hydrograph_review
importlib.reload(hydrograph_review)
from rapid_tools.current_review import collapse_hydrograph_overlay, select_network_pairs

overlay_networks, overlay_omitted = select_network_pairs(
    analysis_df, NETWORK_EXAMPLE, units=UNIT_SELECTION,
    modes=NETWORK_MODES, max_pairs=MAX_NETWORK_PAIRS,
)
print(f"Overlay: {len(overlay_networks)} collapse states; limit {MAX_NETWORK_PAIRS}; omitted {len(overlay_omitted)}.")
display(overlay_networks[["mode", "state_id", "target_id", "hydraulic_domain_type"]])
if len(overlay_networks):
    collapse_hydrograph_overlay(overlay_networks, show_forcing=SHOW_INPUT_FORCING).show()
else:
    print("No collapse states selected.")


Overlay: 1 collapse states; limit 1; omitted 0.


,mode,state_id,target_id,hydraulic_domain_type
0,island,island_15_047,island_15_047,Current whole-network output


## Shared review helpers

These helpers define the three lag classes, compact distributions, stable review indices, and the reusable state inspector.

In [9]:
# import plotly.express as px

# # Shared state-level review frame. Timing classes are mutually exclusive and use
# # the stored percentage shift relative to the base state.
# investigation_df = plot_df.copy()
# lag_metric = "lag_to_inflow_peak_pct_shift_vs_base"
# EQ_METRIC = "Lambda_matched"
# investigation_df[lag_metric] = pd.to_numeric(investigation_df[lag_metric], errors="coerce")
# investigation_df[EQ_METRIC] = pd.to_numeric(investigation_df[EQ_METRIC], errors="coerce")
# investigation_df["neg_log10_lambda"] = -np.log10(investigation_df[EQ_METRIC].where(lambda s: s > 0))

# compact_columns = [
#     "review_index", "example_id", "mode", "state_id", "target_id", "plot_category",
#     lag_metric, EQ_METRIC, "neg_log10_lambda", "hydraulic_domain_type",
#     "parent_domain_n_links", "child_domain_n_links", "length_fraction",
#     "width_fraction_of_total",
# ]

# def _lag_subset_index_plot(frame, *, title, index_value):
#     fig = px.scatter(
#         frame, x="review_index", y=lag_metric, color="mode", symbol="plot_category",
#         hover_data=[c for c in compact_columns if c in frame.columns and c != "review_index"],
#         template="plotly_white", title=title,
#     )
#     fig.update_traces(marker=dict(size=10, line=dict(width=0.7, color="black")))
#     fig.add_hline(y=0, line_dash="dash", line_color="grey")
#     fig.add_vline(x=index_value, line_dash="dot", line_color="black")
#     fig.update_xaxes(dtick=1, title="subset index (change the REVIEW_INDEX below)")
#     fig.update_yaxes(title="lag to inflow peak shift vs base [%]")
#     return fig

# def _eq_distribution(frame, *, title):
#     fig = px.histogram(
#         frame, x=EQ_METRIC, color="mode", facet_col="plot_category",
#         barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title=title,
#         hover_data=[c for c in ["example_id", "state_id", "target_id", lag_metric] if c in frame],
#     )
#     fig.update_xaxes(type="log", title="Lambda_K (selected hydraulic domain; log scale)")
#     fig.update_yaxes(title="state count")
#     return fig



## Shared state inspector

The inspector is called by each section below. It shows the selected hydraulic domain when available; otherwise it explicitly falls back to the full-network geometry view.

In [10]:
# import geopandas as gpd
# from plotly.subplots import make_subplots
# def _add_link_layer(fig, links, *, name, color, width, opacity=1.0, row=1, col=1):
#     xs, ys = [], []
#     for geometry in links.geometry.dropna():
#         geometries = list(geometry.geoms) if geometry.geom_type == "MultiLineString" else [geometry]
#         for line in geometries:
#             x, y = line.xy
#             xs.extend(x); ys.extend(y); xs.append(None); ys.append(None)
#     if xs:
#         fig.add_trace(go.Scattergl(x=xs, y=ys, mode="lines", line=dict(color=color, width=width), opacity=opacity, name=name, hoverinfo="skip"), row=row, col=col)


# def inspect_collapsed_state(example_id, mode, state_id):
#     row=analysis_df[(analysis_df.example_id==example_id)&(analysis_df['mode']==mode)&(analysis_df.state_id==state_id)].iloc[0]
#     child_run=Path(row.refreshed_run_dir);parent_dir=base_directory(example_id)
#     def links(directory):
#         manifest=json.loads((directory/'hydraulic_context/hydraulic_context_manifest.json').read_text())
#         return gpd.read_file(manifest['input_paths']['directed_links'])
#     parent=links(parent_dir);child=links(child_run.parent)
#     pids=[int(x) for x in row.matched_base_link_ids.split(',')];cids=[int(x) for x in row.child_link_ids.split(',')]
#     fig=make_subplots(rows=1,cols=2,subplot_titles=['Refreshed comparison domain','Parent and child outlet'])
#     _add_link_layer(fig,parent,name='Parent network',color='#cbd5e1',width=1)
#     _add_link_layer(fig,parent[parent.id_link.isin(pids)],name='Parent domain',color='#2563eb',width=3)
#     _add_link_layer(fig,child[child.id_link.isin(cids)],name='Child domain',color='#dc2626',width=3)
#     for directory,name,color in [(parent_dir/'run','Parent','#2563eb'),(child_run,'Child','#dc2626')]:
#         h=pd.read_csv(directory/'outlet_hydrograph.csv');t,_=event_clock(example_id,h)
#         fig.add_trace(go.Scatter(x=t,y=h.q_outlet_cms,name=name,line=dict(color=color)),row=1,col=2)
#     fig.update_layout(title=f'{example_id} / {state_id}: {row.hydraulic_domain_type}',height=550,template='plotly_white')
#     fig.update_yaxes(scaleanchor='x',scaleratio=1,row=1,col=1)
#     fig.update_xaxes(title_text='Time (UTC)',row=1,col=2);fig.update_yaxes(title_text='Discharge (m³/s)',row=1,col=2);fig.show()
#     display(row[['Lambda_matched','domain_expanded','parent_domain_n_links','child_domain_n_links','peak_time_shift_hours','interpolated_peak_shift_hours','peak_interpolation_status','event_duration_change_pct','peak_excess_change_pct']].to_frame('value'))


## 1. Zero lag shift — timing ties

These states have exactly zero stored shift in lag to the inflow peak. The purpose is to assess their selected-domain equivalent-resistance distribution and inspect individual ties, not to treat them as delayed responses.

In [11]:
# ZERO_LAG_REVIEW_INDEX = 2
# zero_lag_cases = investigation_df.loc[investigation_df[lag_metric].eq(0)].sort_values(
#     ["example_id", "mode", "state_id"]
# ).reset_index(drop=True)
# zero_lag_cases.insert(0, "review_index", zero_lag_cases.index)
# if zero_lag_cases.empty:
#     raise ValueError("No zero lag-shift states were found.")
# if not 0 <= ZERO_LAG_REVIEW_INDEX < len(zero_lag_cases):
#     raise IndexError(f"ZERO_LAG_REVIEW_INDEX must be between 0 and {len(zero_lag_cases) - 1}")
# zero_lag_selected = zero_lag_cases.iloc[ZERO_LAG_REVIEW_INDEX]

# display(pd.DataFrame({"zero_lag_states": [len(zero_lag_cases)], "all_positive_or_zero": [int(investigation_df[lag_metric].ge(0).sum())]}))
# _eq_distribution(zero_lag_cases, title="Zero lag shift: selected-domain Lambda_K distribution").show()
# _lag_subset_index_plot(zero_lag_cases, title=f"Zero lag shift candidates (n={len(zero_lag_cases)})", index_value=ZERO_LAG_REVIEW_INDEX).show()
# display(zero_lag_selected[[c for c in compact_columns if c in zero_lag_selected.index]].to_frame("selected zero-lag state"))
# inspect_collapsed_state(str(zero_lag_selected.example_id), str(zero_lag_selected['mode']), str(zero_lag_selected.state_id))


## 2. Positive lag shift — delayed outlet peak

These states have a positive relative lag shift. This section shows the lag distribution and selected-domain resistance-ratio distribution, then lets you inspect one state at a time.

In [12]:
# POSITIVE_LAG_REVIEW_INDEX = 0
# positive_lag_cases = investigation_df.loc[investigation_df[lag_metric].gt(0)].sort_values(
#     [lag_metric, "example_id", "mode", "state_id"]
# ).reset_index(drop=True)
# positive_lag_cases.insert(0, "review_index", positive_lag_cases.index)
# if positive_lag_cases.empty:
#     raise ValueError("No positive lag-shift states were found.")
# if not 0 <= POSITIVE_LAG_REVIEW_INDEX < len(positive_lag_cases):
#     raise IndexError(f"POSITIVE_LAG_REVIEW_INDEX must be between 0 and {len(positive_lag_cases) - 1}")
# positive_lag_selected = positive_lag_cases.iloc[POSITIVE_LAG_REVIEW_INDEX]

# display(pd.DataFrame({"positive_lag_states": [len(positive_lag_cases)], "lag_min_pct": [positive_lag_cases[lag_metric].min()], "lag_max_pct": [positive_lag_cases[lag_metric].max()]}))
# px.histogram(positive_lag_cases, x=lag_metric, color="mode", facet_col="plot_category", barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title="Positive lag shift distribution").update_xaxes(title="lag to inflow peak shift vs base [%]").update_yaxes(title="state count").show()
# _eq_distribution(positive_lag_cases, title="Positive lag shift: selected-domain Lambda_K distribution").show()
# _lag_subset_index_plot(positive_lag_cases, title=f"Positive lag shift candidates (n={len(positive_lag_cases)})", index_value=POSITIVE_LAG_REVIEW_INDEX).show()
# display(positive_lag_selected[[c for c in compact_columns if c in positive_lag_selected.index]].to_frame("selected positive-lag state"))
# inspect_collapsed_state(str(positive_lag_selected.example_id), str(positive_lag_selected['mode']), str(positive_lag_selected.state_id))


In [13]:
import importlib
import rapid_tools.current_review as review
importlib.reload(review)

<module 'rapid_tools.current_review' from '/Users/6256481/Code/river-hierarchy/RAPID/src/rapid_tools/current_review.py'>

## 3. Negative lag shift — earlier outlet peak

These states have a negative relative lag shift. They are kept separate because their physical interpretation differs from delayed peak timing; use the same distributions and inspector to review them.

In [14]:
# NEGATIVE_LAG_REVIEW_INDEX = 0
# negative_lag_cases = investigation_df.loc[investigation_df[lag_metric].lt(0)].sort_values(
#     [lag_metric, "example_id", "mode", "state_id"]
# ).reset_index(drop=True)
# negative_lag_cases.insert(0, "review_index", negative_lag_cases.index)
# if negative_lag_cases.empty:
#     raise ValueError("No negative lag-shift states were found.")
# if not 0 <= NEGATIVE_LAG_REVIEW_INDEX < len(negative_lag_cases):
#     raise IndexError(f"NEGATIVE_LAG_REVIEW_INDEX must be between 0 and {len(negative_lag_cases) - 1}")
# negative_lag_selected = negative_lag_cases.iloc[NEGATIVE_LAG_REVIEW_INDEX]

# display(pd.DataFrame({"negative_lag_states": [len(negative_lag_cases)], "lag_min_pct": [negative_lag_cases[lag_metric].min()], "lag_max_pct": [negative_lag_cases[lag_metric].max()]}))
# px.histogram(negative_lag_cases, x=lag_metric, color="mode", facet_col="plot_category", barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title="Negative lag shift distribution").update_xaxes(title="lag to inflow peak shift vs base [%]").update_yaxes(title="state count").show()
# _eq_distribution(negative_lag_cases, title="Negative lag shift: selected-domain Lambda_K distribution").show()
# _lag_subset_index_plot(negative_lag_cases, title=f"Negative lag shift candidates (n={len(negative_lag_cases)})", index_value=NEGATIVE_LAG_REVIEW_INDEX).show()
# display(negative_lag_selected[[c for c in compact_columns if c in negative_lag_selected.index]].to_frame("selected negative-lag state"))
# inspect_collapsed_state(str(negative_lag_selected.example_id), str(negative_lag_selected['mode']), str(negative_lag_selected.state_id))


## Peak-window ambiguity across all 16 cases — 5 October 2026

This audit leaves routing and primary metrics unchanged. Exact maxima, near-peak blocks and normalization sensitivities are retained separately. A maximum **span may contain gaps**; its midpoint is only a diagnostic. The 99/99.5% discharge windows are not confidence intervals or event-duration thresholds. Case 05’s green window is the user-proposed earlier feature, not an adopted replacement peak or proof of lateral inflow. A representative peak rule must be independent of model agreement.

Changing observed peak choices affects validation and gauge-scaled speed; changing the forcing reference time also rescales model percentage lag. Absolute parent–child timing remains unchanged when only those references change.

In [15]:
# # The Excel workbook is now the primary case-review record.
# from IPython.display import FileLink
# display(FileLink(str(REPO_ROOT / "RAPID/reviews/hydrograph_match_review.xlsx")))
